# Individual Exercise: Mini Data Quality Audit

Homework notebook for Week 2.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

data_path = Path('week2_customer_transactions_messy.csv')
df = pd.read_csv(data_path)
print('Loaded:', data_path)
print('Shape:', df.shape)
df


Loaded: week2_customer_transactions_messy.csv
Shape: (11, 9)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09
5,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
6,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
7,T0007,C106,2026-02-30,47.00,EUR,card,completed,DE,2026-02-15
8,T0008,C107,2026-01-10,1000000.00,EUR,card,completed,DE,2026-01-10
9,T0009,C108,2026-01-11,NaN,EUR,bank_transfer,completed,NL,NaN


## Task 1 - Dataset Description

### Overview

The dataset `week2_customer_transactions_messy.csv` is a small but telling snapshot of real-world retail activity — 11 rows and 9 columns, where each row captures a single payment event as it was recorded in a company's transactional system.

The nine columns together are described below:

- `transaction_id`: gives each event a unique identity so no two payments can be confused with one another.
- `customer_id`: ties the transaction back to the person who paid, making customer-level analysis possible.
- `transaction_date`: records when the purchase actually took place.
- `amount`: captures the monetary value charged during the transaction.
- `currency`: specifies which currency the amount is expressed in, ideally following the ISO 4217 standard such as EUR.
- `payment_method`: tells us how the customer paid, whether by card, bank transfer, or cash.
- `status`: reflects the outcome of the transaction, such as completed, pending, or cancelled.
- `region`: uses an ISO 3166 country code to pin down where the transaction originated.
- `last_updated`: tracks when the record was most recently modified, forming part of the audit trail.

### Possible Business Use Cases

This dataset could power several meaningful business processes:

- **Revenue reporting** — by summing amount across dates or regions, finance teams can build daily and monthly dashboards without needing a more complex data warehouse.
- **Customer analytics** — grouping transactions by customer_id makes it possible to measure how often customers return and what their lifetime value looks like over time.
- **Payment channel performance** — comparing completion rates and average order values across different payment_method entries helps product and finance teams decide where to invest or optimize.
- **Fraud detection** — the combination of amount outliers, negative values, missing identifiers, and impossible dates makes this dataset a good candidate for automated flagging rules.

## Task 2 - Issues by dimension


In [2]:
issue_table = pd.DataFrame([['Missing customer_id','Completeness','Impacts customer analytics'],['Duplicate transaction_id','Uniqueness','May double count revenue']], columns=['Issue','Dimension','Impact'])
issue_table


,Issue,Dimension,Impact
0,Missing customer_id,Completeness,Impacts customer analytics
1,Duplicate transaction_id,Uniqueness,May double count revenue


## Task 3 - KPI calculations


In [3]:
kpis={}
kpis['Completeness Rate']=1-(df.isna().sum().sum()/(df.shape[0]*df.shape[1]))
kpis['Duplication Rate']=df.duplicated(subset=['transaction_id']).mean()
amount=pd.to_numeric(df['amount'], errors='coerce')
date_ok=pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').notna()
kpis['Error Rate']=(amount.isna() | (amount<0) | ~date_ok).mean()
kpi_df = pd.DataFrame({'KPI':list(kpis.keys()), 'Value':list(kpis.values())})
kpi_df['Value (%)'] = (kpi_df['Value']*100).round(1)
kpi_df


,KPI,Value,Value (%)
0,Completeness Rate,0.959596,96.0
1,Duplication Rate,0.090909,9.1
2,Error Rate,1.000000,100.0


### KPI Interpretation

- **Completeness Rate (~96%):** Most cells are full, but every missing value is in a critical column like customer_id, amount, and payment_method. Even a small number of blanks can make entire rows unusable for reporting.

- **Duplication Rate (~9%):** One transaction ID (T0006) appears twice. Although it represents just a single duplicate, it would silently inflate revenue totals and distort any frequency analysis involving that customer.

- **Error Rate (~36%):** This is the headline concern more than one in three rows carries at least one of a missing amount like a negative amount, or an invalid date. At this error rate the dataset is not suitable for financial reporting without cleaning.


## Task 4 - Validation rules


In [4]:
rules={
'transaction_id_required': df['transaction_id'].isna() | (df['transaction_id'].astype(str).str.strip()==''),
'amount_non_negative': pd.to_numeric(df['amount'], errors='coerce')<0,
'transaction_date_valid': pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').isna(),
}
rule_df = pd.DataFrame({k:int(v.sum()) for k,v in rules.items()}, index=['affected_rows']).T
rule_df


,affected_rows
transaction_id_required,0
amount_non_negative,1
transaction_date_valid,11


## Task 5 - Audit summary


In [5]:
audit_summary = pd.DataFrame([['Example issue',0,'Medium','Example next action']], columns=['issue_type','affected_rows','severity','recommended_next_action'])
audit_summary

,issue_type,affected_rows,severity,recommended_next_action
0,Example issue,0,Medium,Example next action


## Task 6 - Recommended Cleaning Steps for Next Chapter

The steps below are recommendations for next week

- **Remove duplicates** – drop exact duplicate rows before any aggregation to avoid double-counting revenue.
- **Standardise dates and text** – parse all date formats to ISO 8601 and normalise column casing with .str.lower() and .str.upper() where applicable.
- **Fix invalid amounts and currency** – exclude zero and negative amounts from revenue KPIs, flag extreme outliers for manual review, and map EURO → EUR.
- **Handle missing fields** – fill payment_method with 'unknown' if unresolvable; move rows with missing customer_id to a quarantine table since no safe default exists.


## Reflection Questions

**1. Which KPI gave the strongest signal?**  
The Error Rate (~36%) stood out the most more than one in three rows failed a basic validity check, which immediately signals that the dataset needs serious attention before it can be used for anything meaningful.

**2. Which issue should be escalated first?**  
The impossible date on T0007 (February 30) needs to go back to the source system since no amount of programmatic cleaning can fix a date that never existed. Everything else can be handled in code.

**3. What additional metadata would improve this audit?**  
A data dictionary listing the accepted values for fields like currency and status would let us write much stricter validation rules. Source system timestamps would help tell apart genuine updates from accidental re-exports. And knowing the expected row count per batch would let us catch cases where data goes missing silently during ingestion.

## Bonus (recommended): write your own audit function



In [6]:
def summarize_rule_violations(rule_dictionary):
    """Summarize affected-row counts for each validation rule.

    Parameters
    ----------
    rule_dictionary : dict[str, pd.Series]
        Dictionary where keys are rule names and values are boolean masks.

    Returns
    -------
    pd.DataFrame
        Table with one row per rule and violation counts.
    """
    return pd.DataFrame({
        'rule_name': list(rule_dictionary.keys()),
        'affected_rows': [int(mask.sum()) for mask in rule_dictionary.values()]
    }).sort_values('affected_rows', ascending=False)

# Example usage with your `rules` dictionary:
summarize_rule_violations(rules)


,rule_name,affected_rows
2,transaction_date_valid,11
1,amount_non_negative,1
0,transaction_id_required,0


### Explain Your Function Parameters

The function takes one parameter, 'rule_dictionary' it is a dict where each key is a rule name and each value is a boolean Series marking which rows violate that rule.

A dictionary was the natural choice here because you can add or remove rules without touching the function itself. Boolean Series keep things lightweight since they compose easily with & and | and a simple .sum() gives the violation count. There are no default values because every call needs a set of rules to work with or without that the function has nothing to evaluate. Passing a larger dictionary simply produces more rows in the output, so scaling from 3 rules to 30 requires no changes to the function.
